# Initial Data Profiling

## Objective

Profile the public retail sales dataset before data preparation,
forecasting, or business analysis.

The profiling stage will assess:

- File structure
- Dataset size
- Schema
- Data types
- Date coverage
- Missing values
- Duplicate records
- Retail dimensions
- Dataset grain
- Forecasting suitability

No data cleaning or forecasting is performed in this notebook.

In [1]:
from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parents[1]

# Raw data directory
RAW_DATA = PROJECT_ROOT / "data" / "raw"

# Display project and data locations
print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA)

# Check that the main files exist
for file_name in [
    "train.csv",
    "test.csv",
    "transactions.csv",
    "stores.csv",
    "holidays_events.csv",
    "oil.csv",
]:
    file_path = RAW_DATA / file_name
    print(f"{file_name}: {'FOUND' if file_path.exists() else 'MISSING'}")

Project root: d:\GitHub\retail-sales-forecasting
Raw data: d:\GitHub\retail-sales-forecasting\data\raw
train.csv: FOUND
test.csv: FOUND
transactions.csv: FOUND
stores.csv: FOUND
holidays_events.csv: FOUND
oil.csv: FOUND


## 2.3C — Schema and Data Types

This step inspects the structure of the main public datasets.

We will not modify the source files.

The purpose is to understand the actual schema before designing
data-quality rules, transformations, SQL tables, and forecasting logic.

In [2]:
# Load the main datasets for structural inspection.
# We are only inspecting the schema at this stage.

train_sample = pd.read_csv(
    RAW_DATA / "train.csv",
    nrows=5
)

test_sample = pd.read_csv(
    RAW_DATA / "test.csv",
    nrows=5
)

transactions_sample = pd.read_csv(
    RAW_DATA / "transactions.csv",
    nrows=5
)

stores_sample = pd.read_csv(
    RAW_DATA / "stores.csv",
    nrows=5
)

holidays_sample = pd.read_csv(
    RAW_DATA / "holidays_events.csv",
    nrows=5
)

oil_sample = pd.read_csv(
    RAW_DATA / "oil.csv",
    nrows=5
)

print("TRAIN COLUMNS")
print(train_sample.columns.tolist())

print("\nTRAIN DATA TYPES")
print(train_sample.dtypes)

print("\nTRANSACTIONS COLUMNS")
print(transactions_sample.columns.tolist())

print("\nSTORES COLUMNS")
print(stores_sample.columns.tolist())

print("\nHOLIDAYS COLUMNS")
print(holidays_sample.columns.tolist())

print("\nOIL COLUMNS")
print(oil_sample.columns.tolist())

TRAIN COLUMNS
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']

TRAIN DATA TYPES
id               int64
date               str
store_nbr        int64
family             str
sales          float64
onpromotion      int64
dtype: object

TRANSACTIONS COLUMNS
['date', 'store_nbr', 'transactions']

STORES COLUMNS
['store_nbr', 'city', 'state', 'type', 'cluster']

HOLIDAYS COLUMNS
['date', 'type', 'locale', 'locale_name', 'description', 'transferred']

OIL COLUMNS
['date', 'dcoilwtico']


## 2.3D — Dataset Size

Measure the number of records in the main source files.

This establishes the scale of the dataset before deeper
data-quality and time-series analysis.

In [3]:
# Count rows in each source CSV file.
# We use a lightweight chunked approach for the larger files.

def count_csv_rows(file_path, chunk_size=100_000):
    """Count data rows in a CSV without loading the full file into memory."""
    
    row_count = 0
    
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        row_count += len(chunk)
    
    return row_count


files_to_count = [
    "train.csv",
    "test.csv",
    "transactions.csv",
    "stores.csv",
    "holidays_events.csv",
    "oil.csv",
]

row_counts = {}

for file_name in files_to_count:
    file_path = RAW_DATA / file_name
    
    row_counts[file_name] = count_csv_rows(file_path)
    
    print(f"{file_name}: {row_counts[file_name]:,} rows")

train.csv: 3,000,888 rows
test.csv: 28,512 rows
transactions.csv: 83,488 rows
stores.csv: 54 rows
holidays_events.csv: 350 rows
oil.csv: 1,218 rows


## 2.3E — Historical Date Coverage

Determine the historical period covered by the training data.

This is required before selecting the forecasting grain and horizon.

We will also inspect the number of unique sales dates,
stores, and product families.

In [4]:
# Read only the columns required for initial coverage profiling.
# We avoid loading unnecessary columns from the 3-million-row dataset.

coverage_df = pd.read_csv(
    RAW_DATA / "train.csv",
    usecols=["date", "store_nbr", "family"]
)

# Convert the date column for profiling only.
coverage_df["date"] = pd.to_datetime(coverage_df["date"])

print("Earliest sales date:", coverage_df["date"].min().date())
print("Latest sales date:", coverage_df["date"].max().date())

print("\nUnique sales dates:", coverage_df["date"].nunique())
print("Unique stores:", coverage_df["store_nbr"].nunique())
print("Unique product families:", coverage_df["family"].nunique())

Earliest sales date: 2013-01-01
Latest sales date: 2017-08-15

Unique sales dates: 1684
Unique stores: 54
Unique product families: 33


## 2.3F — Date Continuity

Check whether the historical sales data contains every calendar date
between the minimum and maximum sales dates.

Missing dates can affect time-series aggregation and forecasting,
so they must be identified before model development.

In [5]:
# Get the unique dates from the training data.
sales_dates = coverage_df["date"].drop_duplicates().sort_values()

# Create the complete calendar date range.
expected_dates = pd.date_range(
    start=sales_dates.min(),
    end=sales_dates.max(),
    freq="D"
)

# Identify dates expected in the calendar but absent from the sales data.
missing_dates = expected_dates.difference(sales_dates)

print("Expected calendar dates:", len(expected_dates))
print("Observed sales dates:", len(sales_dates))
print("Missing calendar dates:", len(missing_dates))

print("\nFirst 20 missing dates:")
print(missing_dates[:20])

Expected calendar dates: 1688
Observed sales dates: 1684
Missing calendar dates: 4

First 20 missing dates:
DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'], dtype='datetime64[us]', freq=None)


## 2.3G — Dataset Grain Validation

Determine the lowest logical level represented by the sales dataset.

Initial hypothesis:

> One record per Date + Store + Product Family

We will test whether this combination uniquely identifies
each sales record.

In [6]:
# Load only the columns required to test the dataset grain.
grain_df = pd.read_csv(
    RAW_DATA / "train.csv",
    usecols=["date", "store_nbr", "family", "id"]
)

# Count duplicate combinations of the proposed grain.
duplicate_grain_rows = grain_df.duplicated(
    subset=["date", "store_nbr", "family"]
).sum()

# Count unique combinations of the proposed grain.
unique_grain_combinations = grain_df[
    ["date", "store_nbr", "family"]
].drop_duplicates().shape[0]

# Total records.
total_records = len(grain_df)

print("Total records:", f"{total_records:,}")
print(
    "Unique Date + Store + Family combinations:",
    f"{unique_grain_combinations:,}"
)
print(
    "Duplicate Date + Store + Family records:",
    f"{duplicate_grain_rows:,}"
)

Total records: 3,000,888
Unique Date + Store + Family combinations: 3,000,888
Duplicate Date + Store + Family records: 0


## 2.3H — Record Identifier Validation

Assess whether the `id` column uniquely identifies each sales record.

This helps distinguish a technical row identifier from
the actual business grain of the dataset.

In [7]:
# Validate the uniqueness and range of the source record ID.

id_count = grain_df["id"].count()
unique_id_count = grain_df["id"].nunique()
duplicate_id_count = grain_df["id"].duplicated().sum()

print("Total ID values:", f"{id_count:,}")
print("Unique ID values:", f"{unique_id_count:,}")
print("Duplicate IDs:", f"{duplicate_id_count:,}")

print("\nMinimum ID:", grain_df["id"].min())
print("Maximum ID:", grain_df["id"].max())

Total ID values: 3,000,888
Unique ID values: 3,000,888
Duplicate IDs: 0

Minimum ID: 0
Maximum ID: 3000887


## 2.3I — Missing Value Profiling

Measure missing values in the main sales dataset.

For each column, calculate:

- Total missing values
- Missing percentage

No missing values will be imputed or removed at this stage.
This is profiling only.

In [8]:
# Load the main sales dataset.
# At this stage we need all five analytical columns.

sales_df = pd.read_csv(
    RAW_DATA / "train.csv",
    usecols=[
        "id",
        "date",
        "store_nbr",
        "family",
        "sales",
        "onpromotion"
    ]
)

# Calculate missing-value counts.
missing_counts = sales_df.isna().sum()

# Calculate missing-value percentages.
missing_percentages = (
    sales_df.isna().mean() * 100
)

# Combine the results into a profiling table.
missing_profile = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percentage": missing_percentages
})

print(missing_profile)

             missing_count  missing_percentage
id                       0                 0.0
date                     0                 0.0
store_nbr                0                 0.0
family                   0                 0.0
sales                    0                 0.0
onpromotion              0                 0.0


## 2.3J — Sales Value Validation

Profile the sales measure to identify:

- Distribution statistics
- Zero-sales records
- Negative-sales records

Zero sales are not automatically considered errors.
Negative sales require investigation before any treatment decision.

In [9]:
# Basic statistical profile of the sales measure.

sales_summary = sales_df["sales"].describe()

zero_sales_count = (sales_df["sales"] == 0).sum()
negative_sales_count = (sales_df["sales"] < 0).sum()

print("Sales summary:")
print(sales_summary)

print("\nZero-sales records:", f"{zero_sales_count:,}")
print("Negative-sales records:", f"{negative_sales_count:,}")

Sales summary:
count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

Zero-sales records: 939,130
Negative-sales records: 0


## 2.3K — Promotion Field Validation

Profile the `onpromotion` field to understand how promotional
activity is represented in the sales dataset.

We will determine whether the field behaves like a count,
rather than assuming it is a simple yes/no flag.

In [10]:
# Profile the onpromotion field.

promotion_summary = sales_df["onpromotion"].describe()

zero_promotion_count = (
    sales_df["onpromotion"] == 0
).sum()

positive_promotion_count = (
    sales_df["onpromotion"] > 0
).sum()

negative_promotion_count = (
    sales_df["onpromotion"] < 0
).sum()

print("Promotion summary:")
print(promotion_summary)

print("\nZero-promotion records:", f"{zero_promotion_count:,}")
print("Positive-promotion records:", f"{positive_promotion_count:,}")
print("Negative-promotion records:", f"{negative_promotion_count:,}")

Promotion summary:
count    3.000888e+06
mean     2.602770e+00
std      1.221888e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      7.410000e+02
Name: onpromotion, dtype: float64

Zero-promotion records: 2,389,559
Positive-promotion records: 611,329
Negative-promotion records: 0


## 2.3L — Supporting Dataset: Stores

Validate the store master data before joining it to the sales dataset.

Checks:
- columns and data types
- row count
- missing values
- duplicate store IDs
- store ID coverage
- store type distribution

In [11]:
# Load and profile the store master data.

stores_df = pd.read_csv(
    RAW_DATA / "stores.csv"
)

print("Stores shape:", stores_df.shape)

print("\nColumns:")
print(stores_df.columns.tolist())

print("\nData types:")
print(stores_df.dtypes)

print("\nMissing values:")
print(stores_df.isna().sum())

print("\nDuplicate store IDs:")
print(stores_df["store_nbr"].duplicated().sum())

print("\nStore ID range:")
print("Minimum:", stores_df["store_nbr"].min())
print("Maximum:", stores_df["store_nbr"].max())

print("\nStore types:")
print(stores_df["type"].value_counts().sort_index())

print("\nFirst 5 rows:")
display(stores_df.head())

Stores shape: (54, 5)

Columns:
['store_nbr', 'city', 'state', 'type', 'cluster']

Data types:
store_nbr    int64
city           str
state          str
type           str
cluster      int64
dtype: object

Missing values:
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

Duplicate store IDs:
0

Store ID range:
Minimum: 1
Maximum: 54

Store types:
type
A     9
B     8
C    15
D    18
E     4
Name: count, dtype: int64

First 5 rows:


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


## 2.3M — Store Referential Integrity

Verify that every `store_nbr` appearing in the sales dataset
exists in the store master.

This prevents orphan sales records when the datasets are joined.

In [12]:
# Validate that every sales store exists in the store master.

sales_store_ids = set(sales_df["store_nbr"].unique())
master_store_ids = set(stores_df["store_nbr"].unique())

missing_from_master = sales_store_ids - master_store_ids
unused_master_stores = master_store_ids - sales_store_ids

print("Stores in sales data:", len(sales_store_ids))
print("Stores in store master:", len(master_store_ids))

print("\nSales stores missing from master:", len(missing_from_master))
print("Missing store IDs:", sorted(missing_from_master))

print("\nMaster stores not present in sales:", len(unused_master_stores))
print("Unused master store IDs:", sorted(unused_master_stores))

Stores in sales data: 54
Stores in store master: 54

Sales stores missing from master: 0
Missing store IDs: []

Master stores not present in sales: 0
Unused master store IDs: []


## 2.3N — Supporting Dataset: Transactions

Profile the transaction dataset before using it as a supporting
retail KPI or forecasting driver.

Checks:
- columns and data types
- row count
- missing values
- duplicate Date × Store records
- date coverage
- store coverage
- negative transactions
- zero transactions

In [13]:
# Load and profile the transaction data.

transactions_df = pd.read_csv(
    RAW_DATA / "transactions.csv"
)

print("Transactions shape:", transactions_df.shape)

print("\nColumns:")
print(transactions_df.columns.tolist())

print("\nData types:")
print(transactions_df.dtypes)

print("\nMissing values:")
print(transactions_df.isna().sum())

print("\nDuplicate Date × Store records:")
print(
    transactions_df.duplicated(
        subset=["date", "store_nbr"]
    ).sum()
)

print("\nDate range:")
print("Earliest:", transactions_df["date"].min())
print("Latest:", transactions_df["date"].max())

print("\nUnique dates:", transactions_df["date"].nunique())
print("Unique stores:", transactions_df["store_nbr"].nunique())

print("\nTransaction summary:")
print(transactions_df["transactions"].describe())

print(
    "\nZero-transaction records:",
    (transactions_df["transactions"] == 0).sum()
)

print(
    "Negative-transaction records:",
    (transactions_df["transactions"] < 0).sum()
)

Transactions shape: (83488, 3)

Columns:
['date', 'store_nbr', 'transactions']

Data types:
date              str
store_nbr       int64
transactions    int64
dtype: object

Missing values:
date            0
store_nbr       0
transactions    0
dtype: int64

Duplicate Date × Store records:
0

Date range:
Earliest: 2013-01-01
Latest: 2017-08-15

Unique dates: 1682
Unique stores: 54

Transaction summary:
count    83488.000000
mean      1694.602158
std        963.286644
min          5.000000
25%       1046.000000
50%       1393.000000
75%       2079.000000
max       8359.000000
Name: transactions, dtype: float64

Zero-transaction records: 0
Negative-transaction records: 0


## 2.3O — Transaction and Sales Date Coverage

Compare the unique dates in the sales and transaction datasets.

The purpose is to identify coverage mismatches before the
datasets are joined for retail KPI or forecasting analysis.

In [14]:
# Compare sales and transaction date coverage.

sales_dates = set(
    pd.to_datetime(sales_df["date"]).dt.date
)

transaction_dates = set(
    pd.to_datetime(transactions_df["date"]).dt.date
)

sales_missing_transactions = (
    sales_dates - transaction_dates
)

transactions_missing_sales = (
    transaction_dates - sales_dates
)

print(
    "Sales dates:",
    len(sales_dates)
)

print(
    "Transaction dates:",
    len(transaction_dates)
)

print(
    "\nSales dates missing from transactions:",
    len(sales_missing_transactions)
)

print(
    sorted(sales_missing_transactions)
)

print(
    "\nTransaction dates missing from sales:",
    len(transactions_missing_sales)
)

print(
    sorted(transactions_missing_sales)
)

Sales dates: 1684
Transaction dates: 1682

Sales dates missing from transactions: 2
[datetime.date(2016, 1, 1), datetime.date(2016, 1, 3)]

Transaction dates missing from sales: 0
[]


## 2.3P — Supporting Dataset: Holidays and Events

Profile the holiday/event calendar to understand special dates
that may explain sales and transaction coverage patterns.

Checks:
- columns and data types
- row count
- missing values
- duplicate date/event records
- date coverage
- holiday types
- transferred events

In [15]:
# Load and profile the holiday/event data.

holidays_df = pd.read_csv(
    RAW_DATA / "holidays_events.csv"
)

print("Holidays shape:", holidays_df.shape)

print("\nColumns:")
print(holidays_df.columns.tolist())

print("\nData types:")
print(holidays_df.dtypes)

print("\nMissing values:")
print(holidays_df.isna().sum())

print("\nDate range:")
print("Earliest:", holidays_df["date"].min())
print("Latest:", holidays_df["date"].max())

print("\nUnique dates:", holidays_df["date"].nunique())

print("\nDuplicate full records:")
print(holidays_df.duplicated().sum())

print("\nEvent types:")
print(holidays_df["type"].value_counts().sort_index())

print("\nTransferred events:")
print(holidays_df["transferred"].value_counts(dropna=False))

print("\nFirst 10 rows:")
display(holidays_df.head(10))

Holidays shape: (350, 6)

Columns:
['date', 'type', 'locale', 'locale_name', 'description', 'transferred']

Data types:
date            str
type            str
locale          str
locale_name     str
description     str
transferred    bool
dtype: object

Missing values:
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

Date range:
Earliest: 2012-03-02
Latest: 2017-12-26

Unique dates: 312

Duplicate full records:
0

Event types:
type
Additional     51
Bridge          5
Event          56
Holiday       221
Transfer       12
Work Day        5
Name: count, dtype: int64

Transferred events:
transferred
False    338
True      12
Name: count, dtype: int64

First 10 rows:


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
5,2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,False
6,2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,False
7,2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False
8,2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False
9,2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False


## 2.3Q — Investigate Coverage Anomalies Using the Holiday Calendar

Inspect holiday/event records for dates identified during
sales and transaction coverage validation.

Focus dates:
- December 25, 2013
- December 25, 2014
- December 25, 2015
- December 25, 2016
- January 1, 2016
- January 3, 2016

In [16]:
# Investigate the holiday/event context for the dates
# identified during data-quality profiling.

investigation_dates = [
    "2013-12-25",
    "2014-12-25",
    "2015-12-25",
    "2016-12-25",
    "2016-01-01",
    "2016-01-03",
]

holiday_check = holidays_df[
    holidays_df["date"].isin(investigation_dates)
].sort_values("date")

display(holiday_check)

,date,type,locale,locale_name,description,transferred
89,2013-12-25,Holiday,National,Ecuador,Navidad,False
155,2014-12-25,Holiday,National,Ecuador,Navidad,False
208,2015-12-25,Holiday,National,Ecuador,Navidad,False
211,2016-01-01,Holiday,National,Ecuador,Primer dia del ano,False
294,2016-12-25,Holiday,National,Ecuador,Navidad,False


## 2.3R — Supporting Dataset: Oil Prices

Profile the oil-price dataset before considering it as an
external business/economic driver for retail sales analysis.

Checks:
- columns and data types
- row count
- missing values
- duplicate dates
- date coverage
- zero and negative prices
- price summary

In [17]:
# Load and profile the oil-price data.

oil_df = pd.read_csv(
    RAW_DATA / "oil.csv"
)

print("Oil shape:", oil_df.shape)

print("\nColumns:")
print(oil_df.columns.tolist())

print("\nData types:")
print(oil_df.dtypes)

print("\nMissing values:")
print(oil_df.isna().sum())

print("\nDuplicate dates:")
print(oil_df["date"].duplicated().sum())

print("\nDate range:")
print("Earliest:", oil_df["date"].min())
print("Latest:", oil_df["date"].max())

print("\nUnique dates:", oil_df["date"].nunique())

print("\nOil price summary:")
print(oil_df["dcoilwtico"].describe())

print(
    "\nZero-price records:",
    (oil_df["dcoilwtico"] == 0).sum()
)

print(
    "Negative-price records:",
    (oil_df["dcoilwtico"] < 0).sum()
)

print("\nFirst 10 rows:")
display(oil_df.head(10))

Oil shape: (1218, 2)

Columns:
['date', 'dcoilwtico']

Data types:
date              str
dcoilwtico    float64
dtype: object

Missing values:
date           0
dcoilwtico    43
dtype: int64

Duplicate dates:
0

Date range:
Earliest: 2013-01-01
Latest: 2017-08-31

Unique dates: 1218

Oil price summary:
count    1175.000000
mean       67.714366
std        25.630476
min        26.190000
25%        46.405000
50%        53.190000
75%        95.660000
max       110.620000
Name: dcoilwtico, dtype: float64

Zero-price records: 0
Negative-price records: 0

First 10 rows:


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20
5,2013-01-08,93.21
6,2013-01-09,93.08
7,2013-01-10,93.81
8,2013-01-11,93.60
9,2013-01-14,94.27


## 2.3S — Investigate Missing Oil Prices

Identify the dates where the oil-price value is missing.

The purpose is to determine whether the missing values are
associated with non-trading days or require additional treatment.

In [18]:
# Identify dates with missing oil prices.

missing_oil_dates = oil_df[
    oil_df["dcoilwtico"].isna()
].copy()

print(
    "Number of missing oil-price dates:",
    len(missing_oil_dates)
)

print("\nMissing oil-price dates:")
display(missing_oil_dates)

Number of missing oil-price dates: 43

Missing oil-price dates:


,date,dcoilwtico
0,2013-01-01,NaN
14,2013-01-21,NaN
34,2013-02-18,NaN
63,2013-03-29,NaN
104,2013-05-27,NaN
132,2013-07-04,NaN
174,2013-09-02,NaN
237,2013-11-28,NaN
256,2013-12-25,NaN
261,2014-01-01,NaN


## 2.3T — Validate Missing Oil Dates

Classify missing oil-price dates by day of week.

This helps distinguish expected non-trading-day gaps from
unexpected missing observations.

In [19]:
# Classify missing oil-price dates by day of week.

missing_oil_analysis = missing_oil_dates.copy()

missing_oil_analysis["date"] = pd.to_datetime(
    missing_oil_analysis["date"]
)

missing_oil_analysis["day_of_week"] = (
    missing_oil_analysis["date"]
    .dt.day_name()
)

missing_oil_analysis["day_number"] = (
    missing_oil_analysis["date"]
    .dt.dayofweek
)

print("Missing oil prices by day of week:")
print(
    missing_oil_analysis["day_of_week"]
    .value_counts()
    .sort_index()
)

print("\nTotal missing oil-price dates:")
print(len(missing_oil_analysis))

Missing oil prices by day of week:
day_of_week
Friday        9
Monday       23
Thursday      7
Tuesday       2
Wednesday     2
Name: count, dtype: int64

Total missing oil-price dates:
43


## 2.3U — Transaction Store Referential Integrity

Verify that every store appearing in the transaction dataset
exists in the store master.

In [20]:
# Validate transaction store IDs against the store master.

transaction_store_ids = set(
    transactions_df["store_nbr"].unique()
)

missing_transaction_stores = (
    transaction_store_ids - master_store_ids
)

unused_transaction_stores = (
    master_store_ids - transaction_store_ids
)

print(
    "Stores in transactions:",
    len(transaction_store_ids)
)

print(
    "Stores in store master:",
    len(master_store_ids)
)

print(
    "\nTransaction stores missing from master:",
    len(missing_transaction_stores)
)

print(
    "Missing store IDs:",
    sorted(missing_transaction_stores)
)

print(
    "\nMaster stores not present in transactions:",
    len(unused_transaction_stores)
)

print(
    "Unused store IDs:",
    sorted(unused_transaction_stores)
)

Stores in transactions: 54
Stores in store master: 54

Transaction stores missing from master: 0
Missing store IDs: []

Master stores not present in transactions: 0
Unused store IDs: []


# Phase 2(B) — Overall Data Quality Assessment

## Objective

Evaluate whether the source datasets are sufficiently reliable for
data preparation and retail sales forecasting.

The assessment uses the following data-quality dimensions:

1. Completeness
2. Validity
3. Uniqueness
4. Consistency
5. Referential Integrity
6. Temporal Integrity
7. Business Rule Integrity
8. Cross-Dataset Integrity

Each check will be classified as:

- **PASS** — No issue identified based on the tested rule.
- **REVIEW** — An exception exists and requires documented treatment.
- **FAIL** — A material data-quality problem prevents reliable downstream use.

A PASS does not mean the dataset is perfect. It means the specific
tested rule passed.

## Overall Data Quality Assessment

| Dimension | Check | Result | Status | Treatment / Comment |
|---|---|---|---|---|
| Completeness | Missing values in sales dataset | 0 | PASS | No missing values identified |
| Completeness | Missing values in supporting datasets | Identified in oil | REVIEW | Handle during preparation |
| Validity | Negative sales | 0 | PASS | No negative sales identified |
| Validity | Negative promotion counts | 0 | PASS | No negative promotion counts |
| Validity | Negative transactions | 0 | PASS | No negative transaction counts |
| Uniqueness | Duplicate sales grain | 0 | PASS | Date × Store × Family is unique |
| Uniqueness | Duplicate sales IDs | 0 | PASS | Source IDs are unique |
| Uniqueness | Duplicate transaction grain | 0 | PASS | Date × Store is unique |
| Referential Integrity | Sales → Store master | 0 missing | PASS | All sales stores exist in master |
| Referential Integrity | Transactions → Store master | 0 missing | PASS | All transaction stores exist in master |
| Temporal Integrity | Sales calendar continuity | 4 missing dates | REVIEW | Missing dates require documented treatment |
| Cross-Dataset Integrity | Sales vs Transactions dates | 2 sales dates missing | REVIEW | Investigate before joining |
| Cross-Dataset Integrity | Holiday date multiplicity | Multiple records on some dates | REVIEW | Controlled aggregation/join required |
| Business Rule | onpromotion values | Non-negative count | PASS | Treat as promotion count |
| Business Rule | Oil price values | 43 missing | REVIEW | Treatment required if used as feature |
| Business Rule | Sales grain | Confirmed | PASS | Date × Store × Product Family |

In [21]:
# ============================================================
# Step 2B.3 — Overall Data Quality Scorecard
# ============================================================

# Define the status of each assessed data-quality check.
quality_checks = [
    ("Completeness", "Missing values in sales dataset", "PASS"),
    ("Completeness", "Missing values in supporting datasets", "REVIEW"),
    ("Validity", "Negative sales", "PASS"),
    ("Validity", "Negative promotion counts", "PASS"),
    ("Validity", "Negative transactions", "PASS"),
    ("Uniqueness", "Duplicate sales grain", "PASS"),
    ("Uniqueness", "Duplicate sales IDs", "PASS"),
    ("Uniqueness", "Duplicate transaction grain", "PASS"),
    ("Referential Integrity", "Sales → Store master", "PASS"),
    ("Referential Integrity", "Transactions → Store master", "PASS"),
    ("Temporal Integrity", "Sales calendar continuity", "REVIEW"),
    ("Cross-Dataset Integrity", "Sales vs Transactions dates", "REVIEW"),
    ("Cross-Dataset Integrity", "Holiday date multiplicity", "REVIEW"),
    ("Business Rule", "onpromotion values", "PASS"),
    ("Business Rule", "Oil price values", "REVIEW"),
    ("Business Rule", "Sales grain", "PASS"),
]

# Count each status.
status_counts = {}

for _, _, status in quality_checks:
    status_counts[status] = status_counts.get(status, 0) + 1

# Display the results.
print("Overall Data Quality Scorecard")
print("=" * 40)

print("Total checks:", len(quality_checks))
print("PASS:", status_counts.get("PASS", 0))
print("REVIEW:", status_counts.get("REVIEW", 0))
print("FAIL:", status_counts.get("FAIL", 0))

Overall Data Quality Scorecard
Total checks: 16
PASS: 11
REVIEW: 5
FAIL: 0


In [22]:
# Display every quality check and its status.

for dimension, check, status in quality_checks:
    print(f"[{status}] {dimension} — {check}")

[PASS] Completeness — Missing values in sales dataset
[REVIEW] Completeness — Missing values in supporting datasets
[PASS] Validity — Negative sales
[PASS] Validity — Negative promotion counts
[PASS] Validity — Negative transactions
[PASS] Uniqueness — Duplicate sales grain
[PASS] Uniqueness — Duplicate sales IDs
[PASS] Uniqueness — Duplicate transaction grain
[PASS] Referential Integrity — Sales → Store master
[PASS] Referential Integrity — Transactions → Store master
[REVIEW] Temporal Integrity — Sales calendar continuity
[REVIEW] Cross-Dataset Integrity — Sales vs Transactions dates
[REVIEW] Cross-Dataset Integrity — Holiday date multiplicity
[PASS] Business Rule — onpromotion values
[REVIEW] Business Rule — Oil price values
[PASS] Business Rule — Sales grain


## Data Quality Exceptions Requiring Review

The initial assessment identified five REVIEW areas. These are not
automatically treated as data errors; they require documented treatment
before downstream analysis or forecasting.

### 1. Missing values in supporting datasets

The oil dataset contains **43 missing price observations**.

Treatment decision:
- Preserve the original raw oil data.
- Investigate the missing-date pattern during data preparation.
- Define an appropriate treatment only if oil price is used as a forecasting
  feature.
- Do not modify the raw source file.

### 2. Sales calendar continuity

The sales dataset contains **4 missing calendar dates**:

- 2013-12-25
- 2014-12-25
- 2015-12-25
- 2016-12-25

All four dates have documented national Christmas holiday records
(`Navidad`) in the holiday dataset.

Treatment decision:
- Do not automatically replace the missing dates with zero sales.
- Preserve the source data.
- Account for these calendar exceptions when constructing the forecasting
  time series.

### 3. Sales vs Transactions date coverage

Two sales dates have no corresponding transaction records:

- 2016-01-01
- 2016-01-03

2016-01-01 has a documented national holiday record. 2016-01-03 does not
have a corresponding holiday record in the available holiday dataset.

Treatment decision:
- Do not automatically replace missing transaction values with zero.
- Investigate the dates during data preparation.
- Avoid assuming that missing transaction records represent zero activity.

### 4. Holiday date multiplicity

The holiday/event dataset contains multiple records for some dates.

Treatment decision:
- Do not directly join the raw holiday table to sales at Date × Store ×
  Product Family grain.
- Create a controlled date-level holiday/event representation before joining.
- Preserve the ability to distinguish relevant event characteristics.

### 5. Oil price missing observations

The oil dataset contains **43 missing price observations** and therefore
requires treatment if oil is included as a forecasting feature.

Treatment decision:
- Keep the raw values unchanged.
- Evaluate the missing-date pattern during preparation.
- Choose the treatment based on the forecasting methodology and business
  purpose.
- Document the final treatment and its rationale before model evaluation.

### Overall Quality Conclusion

The initial assessment contains:

- **11 PASS**
- **5 REVIEW**
- **0 FAIL**

The source data is therefore considered **suitable to proceed to structured
data preparation**, subject to documenting and appropriately handling the
identified REVIEW items.

In [23]:
# ============================================================
# Step 2B.6 — Create Machine-Readable Data Quality Results
# ============================================================

import pandas as pd

# Create a structured table from the quality checks.
data_quality_results = pd.DataFrame(
    quality_checks,
    columns=["dimension", "check", "status"]
)

# Add a simple numeric flag for easier reporting later.
data_quality_results["status_flag"] = (
    data_quality_results["status"]
    .map({
        "PASS": 1,
        "REVIEW": 0,
        "FAIL": -1
    })
)

# Display the structured results.
display(data_quality_results)

,dimension,check,status,status_flag
0,Completeness,Missing values in sales dataset,PASS,1
1,Completeness,Missing values in supporting datasets,REVIEW,0
2,Validity,Negative sales,PASS,1
3,Validity,Negative promotion counts,PASS,1
4,Validity,Negative transactions,PASS,1
5,Uniqueness,Duplicate sales grain,PASS,1
6,Uniqueness,Duplicate sales IDs,PASS,1
7,Uniqueness,Duplicate transaction grain,PASS,1
8,Referential Integrity,Sales → Store master,PASS,1
9,Referential Integrity,Transactions → Store master,PASS,1


In [24]:
# Verify the structure and status counts.

print("Rows:", len(data_quality_results))
print("Columns:", list(data_quality_results.columns))

print("\nStatus counts:")
print(
    data_quality_results["status"]
    .value_counts()
    .sort_index()
)

print("\nMissing values:")
print(
    data_quality_results.isna().sum()
)

Rows: 16
Columns: ['dimension', 'check', 'status', 'status_flag']

Status counts:
status
PASS      11
REVIEW     5
Name: count, dtype: int64

Missing values:
dimension      0
check          0
status         0
status_flag    0
dtype: int64


In [25]:
# ============================================================
# Step 2B.9 — Export Data Quality Results
# ============================================================

from pathlib import Path

# Define the validation output path.
validation_dir = Path("../../data/validation")
validation_dir.mkdir(parents=True, exist_ok=True)

quality_results_path = (
    validation_dir / "data_quality_results.csv"
)

# Export the structured quality assessment.
data_quality_results.to_csv(
    quality_results_path,
    index=False
)

print("Exported:", quality_results_path.resolve())
print("Rows exported:", len(data_quality_results))

Exported: D:\GitHub\retail-sales-forecasting\data\validation\data_quality_results.csv
Rows exported: 16


In [26]:
# ============================================================
# Step 2B.10 — Verify Exported Quality Results
# ============================================================

# Read the exported CSV back into Python.
quality_results_check = pd.read_csv(
    quality_results_path
)

print("Rows read back:", len(quality_results_check))
print(
    "Columns:",
    list(quality_results_check.columns)
)

print("\nStatus counts:")
print(
    quality_results_check["status"]
    .value_counts()
    .sort_index()
)

print("\nMissing values:")
print(
    quality_results_check.isna().sum()
)

print("\nFirst 5 rows:")
display(
    quality_results_check.head()
)

Rows read back: 16
Columns: ['dimension', 'check', 'status', 'status_flag']

Status counts:
status
PASS      11
REVIEW     5
Name: count, dtype: int64

Missing values:
dimension      0
check          0
status         0
status_flag    0
dtype: int64

First 5 rows:


,dimension,check,status,status_flag
0,Completeness,Missing values in sales dataset,PASS,1
1,Completeness,Missing values in supporting datasets,REVIEW,0
2,Validity,Negative sales,PASS,1
3,Validity,Negative promotion counts,PASS,1
4,Validity,Negative transactions,PASS,1


# Phase 3 — Data Quality & Preparation

## Step 3.1 — Forecasting Target and Grain

### Forecasting Objective

The initial forecasting objective is to predict **monthly retail sales** at
the total-business level.

### Forecasting Target

The source dataset field `sales` will be used as the forecasting target.

The field will be referred to as **Sales** rather than **Net Sales** because
the source data profile does not establish that `sales` represents net sales.

### Forecasting Grain

The initial forecasting grain is:

**Month × Total Retail Business**

Daily store × product-family sales will therefore be aggregated to monthly
total sales before forecasting.

### Why Monthly Total Sales?

Monthly total sales provides a practical first forecasting layer for:

- Retail management planning
- Sales target setting
- Commercial planning
- Inventory and supply planning
- Budget and business planning

More granular forecasting may be explored later if the data and project
requirements justify it.

### Data Governance Principle

Raw source files will remain unchanged.

All cleaning, transformation, aggregation, and feature engineering will be
performed on working or processed datasets.

# Step 3.2 — Controlled Working Datasets

## Objective

Create controlled working copies of the required raw datasets without
modifying the original source files.

### Data Governance Rule

The `data/raw/` directory is treated as read-only source data.

No cleaning, transformation, aggregation, or feature engineering will be
performed directly on raw files.

Working datasets will be created under:

`data/processed/`

The raw files remain unchanged and can always be used to reproduce the
preparation process.

### Working Datasets

The following datasets will initially be copied into the processed layer:

- `train.csv` → `train_working.csv`
- `transactions.csv` → `transactions_working.csv`
- `stores.csv` → `stores_working.csv`
- `holidays_events.csv` → `holidays_events_working.csv`
- `oil.csv` → `oil_working.csv`

The `test.csv` dataset will remain in the raw layer for now because the
project will use chronological time-series validation rather than treating
the Kaggle test dataset as the primary model validation set.

# Step 3.3 — Data Standardization and Controlled Cleaning Rules

## Objective

Standardize data types and establish controlled cleaning rules for the
working datasets before analytical transformations are performed.

## Data Standardization Rules

The following standardizations will be applied to the working datasets:

- Date fields will be converted to datetime.
- Numeric identifiers will use integer types where appropriate.
- Sales and oil-price measures will use floating-point numeric types.
- Transaction and promotion counts will use integer types.
- Categorical text fields will use string types.
- The holiday `transferred` field will use boolean type where appropriate.

## Controlled Cleaning Rules

### 1. Raw Data Protection

Files in `data/raw/` are treated as read-only source data.

All preparation and cleaning will be performed on datasets in
`data/processed/`.

### 2. Missing Values

Missing values will not be automatically replaced with zero, mean, median,
or another value.

Each missing-value pattern will be evaluated based on its business and
time-series context before treatment.

### 3. Zero Sales

Zero sales observations will be retained unless later validation provides
evidence that a specific observation is invalid.

### 4. Negative Values

Negative sales, transaction counts, and promotion counts will be validated
against the previously completed data-quality results.

No negative values will be removed automatically.

### 5. Outliers

Extreme values will not be removed automatically.

Potential outliers will be investigated using retail context such as stores,
product families, promotions, holidays, and time-series behavior.

### 6. Promotion Counts

`onpromotion` will remain a numeric count because the source data contains
values greater than one.

### 7. Forecasting Target

The forecasting target remains the source field `sales`.

It will be referred to as **Sales**, not **Net Sales**, because the available
source data definition does not establish that `sales` represents net sales.

## Preparation Principle

Standardization changes data representation and type consistency.

Business-rule decisions and analytical transformations will be handled as
separate controlled steps so that each change can be validated and documented.

## Step 3.3 — Train Dataset Type Standardization Test

The training dataset will be loaded from the controlled working layer and
passed through the standardization function.

The test will verify that the expected data types are applied without
changing the dataset's row count.

In [29]:
import pandas as pd

from retail_forecasting.preparation.cleaning import standardize_train_data

train_path = PROJECT_ROOT / "data" / "processed" / "train_working.csv"

train_working = pd.read_csv(train_path)

print("Before standardization:")
print(train_working.dtypes)

train_standardized = standardize_train_data(train_working)

print("\nAfter standardization:")
print(train_standardized.dtypes)

print("\nRow-count validation:")
print("Before:", len(train_working))
print("After: ", len(train_standardized))

Before standardization:
id               int64
date               str
store_nbr        int64
family             str
sales          float64
onpromotion      int64
dtype: object

After standardization:
id                      int64
date           datetime64[us]
store_nbr               int64
family                 string
sales                 float64
onpromotion             int64
dtype: object

Row-count validation:
Before: 3000888
After:  3000888


In [30]:
# ------------------------------------------------------------
# Step 3.3B — Train Dataset Standardization Validation
# ------------------------------------------------------------

expected_dtypes = {
    "id": "int64",
    "date": "datetime64[us]",
    "store_nbr": "int64",
    "family": "string",
    "sales": "float64",
    "onpromotion": "int64",
}

actual_dtypes = train_standardized.dtypes.astype(str).to_dict()

print("Data type validation:")

for column, expected_type in expected_dtypes.items():
    actual_type = actual_dtypes[column]

    print(
        f"{column}: "
        f"expected={expected_type}, "
        f"actual={actual_type}, "
        f"PASS={actual_type == expected_type}"
    )


# ------------------------------------------------------------
# Row-count validation
# ------------------------------------------------------------

row_count_same = len(train_working) == len(train_standardized)

print("\nRow-count validation:")
print(f"PASS={row_count_same}")

Data type validation:
id: expected=int64, actual=int64, PASS=True
date: expected=datetime64[us], actual=datetime64[us], PASS=True
store_nbr: expected=int64, actual=int64, PASS=True
family: expected=string, actual=string, PASS=True
sales: expected=float64, actual=float64, PASS=True
onpromotion: expected=int64, actual=int64, PASS=True

Row-count validation:
PASS=True


## Step 3.3 — Supporting Dataset Standardization

The supporting datasets will be standardized using dedicated preparation
functions.

The validation will confirm:

1. Expected data types are applied.
2. Original row counts are preserved.
3. No missing values are imputed during type standardization.
4. No business-rule transformations are performed at this stage.

In [32]:
# ------------------------------------------------------------
# Step 3.3D — Standardize Supporting Datasets
# ------------------------------------------------------------

import pandas as pd

from retail_forecasting.preparation.cleaning import (
    standardize_holidays_data,
    standardize_oil_data,
    standardize_stores_data,
    standardize_transactions_data,
)

# Load controlled working datasets
transactions_working = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "transactions_working.csv"
)

stores_working = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "stores_working.csv"
)

holidays_working = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "holidays_events_working.csv"
)

oil_working = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "oil_working.csv"
)

# Apply standardization
transactions_standardized = standardize_transactions_data(
    transactions_working
)

stores_standardized = standardize_stores_data(
    stores_working
)

holidays_standardized = standardize_holidays_data(
    holidays_working
)

oil_standardized = standardize_oil_data(
    oil_working
)

# Display resulting data types
print("TRANSACTIONS")
print(transactions_standardized.dtypes)

print("\nSTORES")
print(stores_standardized.dtypes)

print("\nHOLIDAYS / EVENTS")
print(holidays_standardized.dtypes)

print("\nOIL")
print(oil_standardized.dtypes)

TRANSACTIONS
date            datetime64[us]
store_nbr                int64
transactions             int64
dtype: object

STORES
store_nbr     int64
city         string
state        string
type         string
cluster       int64
dtype: object

HOLIDAYS / EVENTS
date           datetime64[us]
type                   string
locale                 string
locale_name            string
description            string
transferred           boolean
dtype: object

OIL
date          datetime64[us]
dcoilwtico           float64
dtype: object


In [33]:
# ------------------------------------------------------------
# Step 3.3E — Row Count and Missing-Value Preservation
# ------------------------------------------------------------

datasets = {
    "train": (train_working, train_standardized),
    "transactions": (
        transactions_working,
        transactions_standardized,
    ),
    "stores": (
        stores_working,
        stores_standardized,
    ),
    "holidays_events": (
        holidays_working,
        holidays_standardized,
    ),
    "oil": (
        oil_working,
        oil_standardized,
    ),
}

print("=" * 70)
print("STEP 3.3E — PRESERVATION VALIDATION")
print("=" * 70)

all_checks_passed = True

for name, (before, after) in datasets.items():

    row_count_same = len(before) == len(after)

    missing_before = before.isna().sum().sum()
    missing_after = after.isna().sum().sum()

    missing_count_same = missing_before == missing_after

    dataset_passed = row_count_same and missing_count_same

    if not dataset_passed:
        all_checks_passed = False

    print(f"\n{name.upper()}")
    print("-" * 40)

    print(
        f"Rows before:        {len(before):,}"
    )

    print(
        f"Rows after:         {len(after):,}"
    )

    print(
        f"Row count preserved: {row_count_same}"
    )

    print(
        f"Missing values before: {missing_before:,}"
    )

    print(
        f"Missing values after:  {missing_after:,}"
    )

    print(
        f"Missing values preserved: {missing_count_same}"
    )

    print(
        f"OVERALL: {'PASS' if dataset_passed else 'FAIL'}"
    )

print("\n" + "=" * 70)
print(
    f"FINAL RESULT: {'PASS' if all_checks_passed else 'FAIL'}"
)
print("=" * 70)

STEP 3.3E — PRESERVATION VALIDATION

TRAIN
----------------------------------------
Rows before:        3,000,888
Rows after:         3,000,888
Row count preserved: True
Missing values before: 0
Missing values after:  0
Missing values preserved: True
OVERALL: PASS

TRANSACTIONS
----------------------------------------
Rows before:        83,488
Rows after:         83,488
Row count preserved: True
Missing values before: 0
Missing values after:  0
Missing values preserved: True
OVERALL: PASS

STORES
----------------------------------------
Rows before:        54
Rows after:         54
Row count preserved: True
Missing values before: 0
Missing values after:  0
Missing values preserved: True
OVERALL: PASS

HOLIDAYS_EVENTS
----------------------------------------
Rows before:        350
Rows after:         350
Row count preserved: True
Missing values before: 0
Missing values after:  0
Missing values preserved: True
OVERALL: PASS

OIL
----------------------------------------
Rows before:    

In [34]:
# ------------------------------------------------------------
# Step 3.3F — Business Value Preservation
# ------------------------------------------------------------

print("=" * 70)
print("STEP 3.3F — BUSINESS VALUE PRESERVATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. TRAIN DATA
# ------------------------------------------------------------

train_sales_same = train_working["sales"].equals(
    train_standardized["sales"]
)

train_promo_same = train_working["onpromotion"].equals(
    train_standardized["onpromotion"]
)

train_store_same = train_working["store_nbr"].equals(
    train_standardized["store_nbr"]
)

train_date_same = (
    pd.to_datetime(train_working["date"]).equals(
        train_standardized["date"]
    )
)

print("\nTRAIN")
print("-" * 40)
print(f"Sales values preserved:       {train_sales_same}")
print(f"Promotion values preserved:   {train_promo_same}")
print(f"Store IDs preserved:          {train_store_same}")
print(f"Dates preserved:              {train_date_same}")


# ------------------------------------------------------------
# 2. TRANSACTIONS
# ------------------------------------------------------------

transactions_values_same = (
    transactions_working["transactions"].equals(
        transactions_standardized["transactions"]
    )
)

transactions_store_same = (
    transactions_working["store_nbr"].equals(
        transactions_standardized["store_nbr"]
    )
)

transactions_date_same = (
    pd.to_datetime(transactions_working["date"]).equals(
        transactions_standardized["date"]
    )
)

print("\nTRANSACTIONS")
print("-" * 40)
print(f"Transaction values preserved: {transactions_values_same}")
print(f"Store IDs preserved:          {transactions_store_same}")
print(f"Dates preserved:              {transactions_date_same}")


# ------------------------------------------------------------
# 3. STORES
# ------------------------------------------------------------

stores_values_same = (
    stores_working["store_nbr"].equals(
        stores_standardized["store_nbr"]
    )
    and stores_working["cluster"].equals(
        stores_standardized["cluster"]
    )
)

stores_text_same = all(
    stores_working[column].astype("string").equals(
        stores_standardized[column]
    )
    for column in ["city", "state", "type"]
)

print("\nSTORES")
print("-" * 40)
print(f"Numeric values preserved:     {stores_values_same}")
print(f"Text values preserved:        {stores_text_same}")


# ------------------------------------------------------------
# 4. HOLIDAYS / EVENTS
# ------------------------------------------------------------

holiday_text_same = all(
    holidays_working[column].astype("string").equals(
        holidays_standardized[column]
    )
    for column in [
        "type",
        "locale",
        "locale_name",
        "description",
    ]
)

holiday_date_same = (
    pd.to_datetime(holidays_working["date"]).equals(
        holidays_standardized["date"]
    )
)

holiday_transfer_same = (
    holidays_working["transferred"].equals(
        holidays_standardized["transferred"]
    )
)

print("\nHOLIDAYS / EVENTS")
print("-" * 40)
print(f"Dates preserved:              {holiday_date_same}")
print(f"Text values preserved:        {holiday_text_same}")
print(f"Transfer values preserved:    {holiday_transfer_same}")


# ------------------------------------------------------------
# 5. OIL
# ------------------------------------------------------------

oil_date_same = (
    pd.to_datetime(oil_working["date"]).equals(
        oil_standardized["date"]
    )
)

oil_values_same = (
    oil_working["dcoilwtico"].equals(
        oil_standardized["dcoilwtico"]
    )
)

print("\nOIL")
print("-" * 40)
print(f"Dates preserved:              {oil_date_same}")
print(f"Oil price values preserved:   {oil_values_same}")


# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

all_business_values_preserved = all([
    train_sales_same,
    train_promo_same,
    train_store_same,
    train_date_same,
    transactions_values_same,
    transactions_store_same,
    transactions_date_same,
    stores_values_same,
    stores_text_same,
    holiday_text_same,
    holiday_date_same,
    holiday_transfer_same,
    oil_date_same,
    oil_values_same,
])

print("\n" + "=" * 70)
print(
    f"FINAL RESULT: "
    f"{'PASS' if all_business_values_preserved else 'FAIL'}"
)
print("=" * 70)

STEP 3.3F — BUSINESS VALUE PRESERVATION

TRAIN
----------------------------------------
Sales values preserved:       True
Promotion values preserved:   True
Store IDs preserved:          True
Dates preserved:              True

TRANSACTIONS
----------------------------------------
Transaction values preserved: True
Store IDs preserved:          True
Dates preserved:              True

STORES
----------------------------------------
Numeric values preserved:     True
Text values preserved:        True

HOLIDAYS / EVENTS
----------------------------------------
Dates preserved:              True
Text values preserved:        True
Transfer values preserved:    False

OIL
----------------------------------------
Dates preserved:              True
Oil price values preserved:   True

FINAL RESULT: FAIL


In [35]:
# ------------------------------------------------------------
# Step 3.3G — Investigate Holiday Transfer Value Difference
# ------------------------------------------------------------

print("=" * 70)
print("STEP 3.3G — TRANSFERRED FIELD INVESTIGATION")
print("=" * 70)

original = holidays_working["transferred"]
standardized = holidays_standardized["transferred"]

print("\nDATA TYPES")
print("-" * 40)
print("Original dtype:     ", original.dtype)
print("Standardized dtype: ", standardized.dtype)

print("\nVALUE COUNTS — ORIGINAL")
print("-" * 40)
print(original.value_counts(dropna=False))

print("\nVALUE COUNTS — STANDARDIZED")
print("-" * 40)
print(standardized.value_counts(dropna=False))

print("\nNULL COUNTS")
print("-" * 40)
print("Original nulls:     ", original.isna().sum())
print("Standardized nulls: ", standardized.isna().sum())

# Compare actual boolean values after converting both
# to the same nullable boolean representation.
original_as_boolean = original.astype("boolean")

value_only_same = original_as_boolean.equals(
    standardized
)

print("\nVALUE-ONLY COMPARISON")
print("-" * 40)
print(
    "Boolean values preserved:",
    value_only_same
)

# Show any actual differences, if present.
differences = original_as_boolean != standardized

print("\nACTUAL VALUE DIFFERENCES")
print("-" * 40)
print("Number of differing rows:", differences.sum())

if differences.sum() > 0:
    print("\nRows with differences:")
    print(
        pd.DataFrame({
            "original": original_as_boolean[differences],
            "standardized": standardized[differences]
        })
    )

STEP 3.3G — TRANSFERRED FIELD INVESTIGATION

DATA TYPES
----------------------------------------
Original dtype:      bool
Standardized dtype:  boolean

VALUE COUNTS — ORIGINAL
----------------------------------------
transferred
False    338
True      12
Name: count, dtype: int64

VALUE COUNTS — STANDARDIZED
----------------------------------------
transferred
False    338
True      12
Name: count, dtype: Int64

NULL COUNTS
----------------------------------------
Original nulls:      0
Standardized nulls:  0

VALUE-ONLY COMPARISON
----------------------------------------
Boolean values preserved: True

ACTUAL VALUE DIFFERENCES
----------------------------------------
Number of differing rows: 0


In [36]:
# ------------------------------------------------------------
# Step 3.3H — Final Standardization Validation
# ------------------------------------------------------------

validation_results = []

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

validation_results.extend([
    ["train", "row_count_preserved",
     len(train_working) == len(train_standardized)],

    ["train", "missing_values_preserved",
     train_working.isna().sum().sum()
     == train_standardized.isna().sum().sum()],

    ["train", "sales_values_preserved",
     train_working["sales"].equals(
         train_standardized["sales"]
     )],

    ["train", "promotion_values_preserved",
     train_working["onpromotion"].equals(
         train_standardized["onpromotion"]
     )],
])


# ------------------------------------------------------------
# TRANSACTIONS
# ------------------------------------------------------------

validation_results.extend([
    ["transactions", "row_count_preserved",
     len(transactions_working)
     == len(transactions_standardized)],

    ["transactions", "missing_values_preserved",
     transactions_working.isna().sum().sum()
     == transactions_standardized.isna().sum().sum()],

    ["transactions", "values_preserved",
     transactions_working["transactions"].equals(
         transactions_standardized["transactions"]
     )],
])


# ------------------------------------------------------------
# STORES
# ------------------------------------------------------------

validation_results.extend([
    ["stores", "row_count_preserved",
     len(stores_working)
     == len(stores_standardized)],

    ["stores", "missing_values_preserved",
     stores_working.isna().sum().sum()
     == stores_standardized.isna().sum().sum()],

    ["stores", "store_ids_preserved",
     stores_working["store_nbr"].equals(
         stores_standardized["store_nbr"]
     )],

    ["stores", "cluster_values_preserved",
     stores_working["cluster"].equals(
         stores_standardized["cluster"]
     )],
])


# ------------------------------------------------------------
# HOLIDAYS / EVENTS
# ------------------------------------------------------------

validation_results.extend([
    ["holidays_events", "row_count_preserved",
     len(holidays_working)
     == len(holidays_standardized)],

    ["holidays_events", "missing_values_preserved",
     holidays_working.isna().sum().sum()
     == holidays_standardized.isna().sum().sum()],

    ["holidays_events", "dates_preserved",
     pd.to_datetime(
         holidays_working["date"]
     ).equals(
         holidays_standardized["date"]
     )],

    ["holidays_events", "text_values_preserved",
     all(
         holidays_working[column]
         .astype("string")
         .equals(holidays_standardized[column])
         for column in [
             "type",
             "locale",
             "locale_name",
             "description",
         ]
     )],

    ["holidays_events", "transferred_values_preserved",
     holidays_working["transferred"]
     .astype("boolean")
     .equals(
         holidays_standardized["transferred"]
     )],
])


# ------------------------------------------------------------
# OIL
# ------------------------------------------------------------

validation_results.extend([
    ["oil", "row_count_preserved",
     len(oil_working)
     == len(oil_standardized)],

    ["oil", "missing_values_preserved",
     oil_working.isna().sum().sum()
     == oil_standardized.isna().sum().sum()],

    ["oil", "oil_values_preserved",
     oil_working["dcoilwtico"].equals(
         oil_standardized["dcoilwtico"]
     )],
])


# ------------------------------------------------------------
# CREATE VALIDATION TABLE
# ------------------------------------------------------------

validation_df = pd.DataFrame(
    validation_results,
    columns=["dataset", "check", "passed"]
)

print("=" * 70)
print("FINAL STANDARDIZATION VALIDATION")
print("=" * 70)

print(validation_df.to_string(index=False))

print("\n" + "=" * 70)

total_checks = len(validation_df)
passed_checks = validation_df["passed"].sum()
failed_checks = total_checks - passed_checks

print(f"Total checks:  {total_checks}")
print(f"Passed:        {passed_checks}")
print(f"Failed:        {failed_checks}")

print(
    "\nFINAL RESULT:",
    "PASS" if failed_checks == 0 else "FAIL"
)

print("=" * 70)

FINAL STANDARDIZATION VALIDATION
        dataset                        check  passed
          train          row_count_preserved    True
          train     missing_values_preserved    True
          train       sales_values_preserved    True
          train   promotion_values_preserved    True
   transactions          row_count_preserved    True
   transactions     missing_values_preserved    True
   transactions             values_preserved    True
         stores          row_count_preserved    True
         stores     missing_values_preserved    True
         stores          store_ids_preserved    True
         stores     cluster_values_preserved    True
holidays_events          row_count_preserved    True
holidays_events     missing_values_preserved    True
holidays_events              dates_preserved    True
holidays_events        text_values_preserved    True
holidays_events transferred_values_preserved    True
            oil          row_count_preserved    True
            o

In [37]:
# ------------------------------------------------------------
# Step 3.3I — Save Standardized Working Datasets
# ------------------------------------------------------------

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_standardized.to_csv(
    PROCESSED_DIR / "train_standardized.csv",
    index=False
)

transactions_standardized.to_csv(
    PROCESSED_DIR / "transactions_standardized.csv",
    index=False
)

stores_standardized.to_csv(
    PROCESSED_DIR / "stores_standardized.csv",
    index=False
)

holidays_standardized.to_csv(
    PROCESSED_DIR / "holidays_events_standardized.csv",
    index=False
)

oil_standardized.to_csv(
    PROCESSED_DIR / "oil_standardized.csv",
    index=False
)

print("Standardized datasets saved successfully:")
print()
print("1.", PROCESSED_DIR / "train_standardized.csv")
print("2.", PROCESSED_DIR / "transactions_standardized.csv")
print("3.", PROCESSED_DIR / "stores_standardized.csv")
print("4.", PROCESSED_DIR / "holidays_events_standardized.csv")
print("5.", PROCESSED_DIR / "oil_standardized.csv")

Standardized datasets saved successfully:

1. d:\GitHub\retail-sales-forecasting\data\processed\train_standardized.csv
2. d:\GitHub\retail-sales-forecasting\data\processed\transactions_standardized.csv
3. d:\GitHub\retail-sales-forecasting\data\processed\stores_standardized.csv
4. d:\GitHub\retail-sales-forecasting\data\processed\holidays_events_standardized.csv
5. d:\GitHub\retail-sales-forecasting\data\processed\oil_standardized.csv


In [38]:
# ------------------------------------------------------------
# Step 3.3J — Disk-Level Verification
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

standardized_files = {
    "train": "train_standardized.csv",
    "transactions": "transactions_standardized.csv",
    "stores": "stores_standardized.csv",
    "holidays_events": "holidays_events_standardized.csv",
    "oil": "oil_standardized.csv",
}

disk_data = {}

print("=" * 70)
print("STEP 3.3J — SAVED FILE VERIFICATION")
print("=" * 70)

for name, filename in standardized_files.items():

    file_path = PROCESSED_DIR / filename

    print(f"\n{name.upper()}")
    print("-" * 40)

    print("File exists:", file_path.exists())

    if file_path.exists():
        df = pd.read_csv(file_path)
        disk_data[name] = df

        print("Rows:", f"{len(df):,}")
        print("Columns:", len(df))
        print("Missing values:", f"{df.isna().sum().sum():,}")

        print("\nData types:")
        print(df.dtypes)

print("\n" + "=" * 70)
print("DISK VERIFICATION COMPLETE")
print("=" * 70)

STEP 3.3J — SAVED FILE VERIFICATION

TRAIN
----------------------------------------
File exists: True
Rows: 3,000,888
Columns: 3000888
Missing values: 0

Data types:
id               int64
date               str
store_nbr        int64
family             str
sales          float64
onpromotion      int64
dtype: object

TRANSACTIONS
----------------------------------------
File exists: True
Rows: 83,488
Columns: 83488
Missing values: 0

Data types:
date              str
store_nbr       int64
transactions    int64
dtype: object

STORES
----------------------------------------
File exists: True
Rows: 54
Columns: 54
Missing values: 0

Data types:
store_nbr    int64
city           str
state          str
type           str
cluster      int64
dtype: object

HOLIDAYS_EVENTS
----------------------------------------
File exists: True
Rows: 350
Columns: 350
Missing values: 0

Data types:
date            str
type            str
locale          str
locale_name     str
description     str
transferred 

In [39]:
# ------------------------------------------------------------
# Step 3.3K — Final Saved-File Structural Validation
# ------------------------------------------------------------

expected_structure = {
    "train": {
        "file": "train_standardized.csv",
        "rows": 3_000_888,
        "columns": [
            "id",
            "date",
            "store_nbr",
            "family",
            "sales",
            "onpromotion",
        ],
        "missing_values": 0,
    },

    "transactions": {
        "file": "transactions_standardized.csv",
        "rows": 83_488,
        "columns": [
            "date",
            "store_nbr",
            "transactions",
        ],
        "missing_values": 0,
    },

    "stores": {
        "file": "stores_standardized.csv",
        "rows": 54,
        "columns": [
            "store_nbr",
            "city",
            "state",
            "type",
            "cluster",
        ],
        "missing_values": 0,
    },

    "holidays_events": {
        "file": "holidays_events_standardized.csv",
        "rows": 350,
        "columns": [
            "date",
            "type",
            "locale",
            "locale_name",
            "description",
            "transferred",
        ],
        "missing_values": 0,
    },

    "oil": {
        "file": "oil_standardized.csv",
        "rows": 1_218,
        "columns": [
            "date",
            "dcoilwtico",
        ],
        "missing_values": 43,
    },
}


results = []

for name, expected in expected_structure.items():

    file_path = PROCESSED_DIR / expected["file"]

    df = pd.read_csv(file_path)

    file_exists = file_path.exists()
    row_count_correct = len(df) == expected["rows"]
    columns_correct = list(df.columns) == expected["columns"]
    missing_values_correct = (
        df.isna().sum().sum()
        == expected["missing_values"]
    )

    dataset_pass = all([
        file_exists,
        row_count_correct,
        columns_correct,
        missing_values_correct,
    ])

    results.append([
        name,
        file_exists,
        row_count_correct,
        columns_correct,
        missing_values_correct,
        dataset_pass,
    ])


validation_df = pd.DataFrame(
    results,
    columns=[
        "dataset",
        "file_exists",
        "row_count_correct",
        "columns_correct",
        "missing_values_correct",
        "overall_pass",
    ],
)

print("=" * 90)
print("STEP 3.3K — FINAL SAVED-FILE STRUCTURAL VALIDATION")
print("=" * 90)

print(validation_df.to_string(index=False))

print("\n" + "=" * 90)

total_datasets = len(validation_df)
passed_datasets = validation_df["overall_pass"].sum()
failed_datasets = total_datasets - passed_datasets

print(f"Datasets checked: {total_datasets}")
print(f"Datasets passed:  {passed_datasets}")
print(f"Datasets failed:  {failed_datasets}")

print(
    "\nFINAL RESULT:",
    "PASS" if failed_datasets == 0 else "FAIL"
)

print("=" * 90)

STEP 3.3K — FINAL SAVED-FILE STRUCTURAL VALIDATION
        dataset  file_exists  row_count_correct  columns_correct  missing_values_correct  overall_pass
          train         True               True             True                    True          True
   transactions         True               True             True                    True          True
         stores         True               True             True                    True          True
holidays_events         True               True             True                    True          True
            oil         True               True             True                    True          True

Datasets checked: 5
Datasets passed:  5
Datasets failed:  0

FINAL RESULT: PASS


# Step 3.3 — Data Standardization Completed

## Objective

Standardize the data types of the controlled working datasets while preserving
the original business values, row counts, and missing-value patterns.

## Standardized Datasets

The following datasets were standardized:

- `train_working.csv`
- `transactions_working.csv`
- `stores_working.csv`
- `holidays_events_working.csv`
- `oil_working.csv`

Standardized versions were saved separately as:

- `train_standardized.csv`
- `transactions_standardized.csv`
- `stores_standardized.csv`
- `holidays_events_standardized.csv`
- `oil_standardized.csv`

## Standardization Rules

### Train

- `id` → integer
- `date` → datetime
- `store_nbr` → integer
- `family` → string
- `sales` → float
- `onpromotion` → integer

### Transactions

- `date` → datetime
- `store_nbr` → integer
- `transactions` → integer

### Stores

- `store_nbr` → integer
- `city` → string
- `state` → string
- `type` → string
- `cluster` → integer

### Holidays / Events

- `date` → datetime
- text fields → string
- `transferred` → boolean

### Oil

- `date` → datetime
- `dcoilwtico` → float

## Validation Results

Standardization was validated before saving the datasets.

### In-memory validation

- 19 validation checks
- 19 PASS
- 0 FAIL

### Saved-file structural validation

- 5 datasets checked
- 5 PASS
- 0 FAIL

The validation confirmed that:

- Row counts were preserved.
- Missing-value counts were preserved.
- Business values were preserved.
- Expected columns were preserved.
- Standardized files were successfully written to `data/processed/`.

The 43 missing oil-price values were intentionally preserved and will
be evaluated separately during business-rule preparation.

## Important Data Governance Decision

The original files in `data/raw/` remain unchanged.

The project maintains separate layers for:

1. Raw source data
2. Controlled working data
3. Standardized data
4. Future cleaned/analytical data

No business-rule cleaning, imputation, outlier removal, holiday aggregation,
or forecasting transformation was performed during this stage.